In [ ]:
!pip install --upgrade synthcity
!pip show synthcity

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchtext torchmetrics torchao torchtune torchtuples
!pip install torch==2.4.0 torchvision==0.19.0 torchtext==0.18.0

In [ ]:
!pip install torchtuples
!pip uninstall numpy -y
!pip install --upgrade "numpy==2.0.0" "networkx>=3.2" 

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/datasets/manuelamarenghi/dataset-ricoveri/dataset_ricoveri_encode_index.csv',
    sep=';',
    quotechar='"',
    encoding='utf-8',
    on_bad_lines='skip'
)

print(df.head())
print(df.columns)

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split

cod_regs = df["COD_REG"].dropna().unique()

train_cod, test_cod = train_test_split(
    cod_regs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = df[df["COD_REG"].isin(train_cod)].copy()
test_df  = df[df["COD_REG"].isin(test_cod)].copy()

In [ ]:
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from synthcity.plugins.core.dataloader import TimeSeriesSurvivalDataLoader


class RicoveriTimeSeriesDataloader:
    """
    Prepara i dati di ricovero per synthcity TimeSeriesDataLoader.
    """

    TEMPORAL_CONTINUOUS = [
        "eta",
        "qt_prest_Sum",
    ]

    TEMPORAL_BINARY = [
        "SHOCK",
        "CABG",
        "PTCA",
        "ICD"
    ]

    TEMPORAL_CATEGORICAL = [
        "class_prest",
    ]

    TIME_INDEX_COL = "time_days"

    STATIC_CONTINUOUS = [
        "MCS_score",
    ]

    STATIC_BINARY = [
        "sesso",
    ]

    STATIC_CATEGORICAL = [
        "gruppo",
    ]

    OUTCOME_EVENT_COL = "dec_intra"
    OUTCOME_TIME_COL = "death_time"

    def __init__(
        self,
        seq_len: int = 5,
        step: int = 5,
        max_windows_per_patient: int = 1,
        balance_deceased: bool = True,
        n_patients: int = 2000,
        random_seed: int = 42,
    ):
        self.seq_len = seq_len
        self.step = step
        self.max_windows = max_windows_per_patient
        self.balance_deceased = balance_deceased
        self.n_patients = n_patients
        self.seed = random_seed

        self.scaler_temp = MinMaxScaler()
        self.scaler_static = MinMaxScaler()
        self.encoder_temp = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.encoder_static = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

        self.imputer_temp_cont = SimpleImputer(strategy="median")
        self.imputer_temp_cat = SimpleImputer(strategy="most_frequent")
        self.imputer_static_cont = SimpleImputer(strategy="median")
        self.imputer_static_cat = SimpleImputer(strategy="most_frequent")

    def _select_patients(self, df: pd.DataFrame) -> pd.DataFrame:
        rng = np.random.default_rng(self.seed)

        eligible = [
            cod for cod, grp in df.groupby("COD_REG")
            if len(grp) > 2 and (grp["desc_studio_out"] != 2).any()
        ]

        deceased = [c for c in eligible if df[df["COD_REG"] == c]["dec_intra"].max() == 1]
        survived = [c for c in eligible if df[df["COD_REG"] == c]["dec_intra"].max() == 0]

        if self.balance_deceased:
            n_dec = min(len(deceased), self.n_patients // 2)
            n_surv = min(len(survived), self.n_patients - n_dec)
        else:
            n_dec = min(len(deceased), self.n_patients)
            n_surv = 0

        chosen_dec = rng.choice(deceased, size=n_dec, replace=False)
        chosen_surv = rng.choice(survived, size=n_surv, replace=False)
        chosen = np.concatenate([chosen_dec, chosen_surv])

        print(f"Pazienti selezionati → deceduti: {n_dec}, sopravvissuti: {n_surv}, totale: {len(chosen)}")
        return df[df["COD_REG"].isin(chosen)].copy()

    def _preprocess_temporal(self, data: pd.DataFrame, fit: bool = True) -> pd.DataFrame:

        if self.TEMPORAL_CONTINUOUS:
            cont = data[self.TEMPORAL_CONTINUOUS].copy()
            if fit:
                cont_imp = pd.DataFrame(
                    self.imputer_temp_cont.fit_transform(cont),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
                cont_scaled = pd.DataFrame(
                    self.scaler_temp.fit_transform(cont_imp),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
            else:
                cont_imp = pd.DataFrame(
                    self.imputer_temp_cont.transform(cont),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
                cont_scaled = pd.DataFrame(
                    self.scaler_temp.transform(cont_imp),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
            data[self.TEMPORAL_CONTINUOUS] = cont_scaled

        if self.TEMPORAL_BINARY:
            bin_imp = SimpleImputer(strategy="most_frequent")
            data[self.TEMPORAL_BINARY] = bin_imp.fit_transform(data[self.TEMPORAL_BINARY])
            data[self.TEMPORAL_BINARY] = data[self.TEMPORAL_BINARY].astype(int)

        if self.TEMPORAL_CATEGORICAL:
            cat = data[self.TEMPORAL_CATEGORICAL].copy()
            if fit:
                cat_imp = pd.DataFrame(
                    self.imputer_temp_cat.fit_transform(cat),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
                cat_enc = pd.DataFrame(
                    self.encoder_temp.fit_transform(cat_imp),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
            else:
                cat_imp = pd.DataFrame(
                    self.imputer_temp_cat.transform(cat),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
                cat_enc = pd.DataFrame(
                    self.encoder_temp.transform(cat_imp),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
            data[self.TEMPORAL_CATEGORICAL] = cat_enc

        return data

    def _preprocess_static(self, static: pd.DataFrame, fit: bool = True) -> pd.DataFrame:

        if self.STATIC_CONTINUOUS:
            cont = static[self.STATIC_CONTINUOUS].copy()
            if fit:
                cont_imp = pd.DataFrame(
                    self.imputer_static_cont.fit_transform(cont),
                    columns=self.STATIC_CONTINUOUS, index=static.index
                )
                static[self.STATIC_CONTINUOUS] = self.scaler_static.fit_transform(cont_imp)
            else:
                cont_imp = pd.DataFrame(
                    self.imputer_static_cont.transform(cont),
                    columns=self.STATIC_CONTINUOUS, index=static.index
                )
                static[self.STATIC_CONTINUOUS] = self.scaler_static.transform(cont_imp)

        if self.STATIC_BINARY:
            bin_imp = SimpleImputer(strategy="most_frequent")
            static[self.STATIC_BINARY] = bin_imp.fit_transform(static[self.STATIC_BINARY])
            static[self.STATIC_BINARY] = static[self.STATIC_BINARY].astype(int)

        if self.STATIC_CATEGORICAL:
            cat = static[self.STATIC_CATEGORICAL].copy()
            if fit:
                cat_imp = pd.DataFrame(
                    self.imputer_static_cat.fit_transform(cat),
                    columns=self.STATIC_CATEGORICAL, index=static.index
                )
                static[self.STATIC_CATEGORICAL] = self.encoder_static.fit_transform(cat_imp)
            else:
                cat_imp = pd.DataFrame(
                    self.imputer_static_cat.transform(cat),
                    columns=self.STATIC_CATEGORICAL, index=static.index
                )
                static[self.STATIC_CATEGORICAL] = self.encoder_static.transform(cat_imp)

        return static

    def _make_windows(
        self,
        group: pd.DataFrame,
        temporal_cols: List[str],
    ) -> Tuple[List[pd.DataFrame], List[np.ndarray]]:

        windows, times = [], []
        n = len(group)

        indices = range(0, max(1, n - self.seq_len + 1), self.step)
        chosen_indices = list(indices)[: self.max_windows]

        for start in chosen_indices:
            end = start + self.seq_len
            chunk = group.iloc[start:end]

            if len(chunk) < self.seq_len:
                pad_rows = self.seq_len - len(chunk)
                pad = pd.concat([chunk.iloc[[-1]]] * pad_rows, ignore_index=True)
                chunk = pd.concat([chunk, pad], ignore_index=True)

            windows.append(chunk[temporal_cols].reset_index(drop=True))
            times.append(chunk[self.TIME_INDEX_COL].values)

        return windows, times

    def inverse_static(self, static_df: pd.DataFrame) -> pd.DataFrame:
        df = static_df.copy()
    
        # CONTINUE
        if self.STATIC_CONTINUOUS:
            df[self.STATIC_CONTINUOUS] = self.scaler_static.inverse_transform(
                df[self.STATIC_CONTINUOUS]
            )
    
        # CATEGORICHE
        if self.STATIC_CATEGORICAL:
            df[self.STATIC_CATEGORICAL] = self.encoder_static.inverse_transform(
                df[self.STATIC_CATEGORICAL]
            )
    
        return df

    def inverse_temporal(self, windows: List[pd.DataFrame]) -> List[pd.DataFrame]:
        inv_windows = []
    
        for w in windows:
            w = w.copy()
    
            # CONTINUE
            if self.TEMPORAL_CONTINUOUS:
                w[self.TEMPORAL_CONTINUOUS] = self.scaler_temp.inverse_transform(
                    w[self.TEMPORAL_CONTINUOUS]
                )
    
            # CATEGORICHE
            if self.TEMPORAL_CATEGORICAL:
                w[self.TEMPORAL_CATEGORICAL] = self.encoder_temp.inverse_transform(
                    w[self.TEMPORAL_CATEGORICAL]
                )
    
            inv_windows.append(w)
    
        return inv_windows
    
    def load(
        self,
        df: pd.DataFrame,
        fit: bool = True,
    ) -> TimeSeriesSurvivalDataLoader:

        data = self._select_patients(df) if fit else df.copy()
        data = data.sort_values(["COD_REG", "time_step"]).reset_index(drop=True)

        temporal_cols = self.TEMPORAL_CONTINUOUS + self.TEMPORAL_BINARY + self.TEMPORAL_CATEGORICAL
        data = self._preprocess_temporal(data, fit=fit)

        all_windows, all_times, static_rows, T_list, E_list, patient_ids  = [], [], [], [], [], []

        for cod, group in data.groupby("COD_REG"):
            if len(group) < 2:
                continue

            wins, times = self._make_windows(group, temporal_cols)

            for w, t in zip(wins, times):
                w = w.fillna(0)  # ✅ FIX TEMPORAL

                all_windows.append(w)
                all_times.append(t)
                patient_ids.append(cod)

                static_row = group[
                    self.STATIC_CONTINUOUS + self.STATIC_BINARY + self.STATIC_CATEGORICAL
                ].iloc[-1]

                static_rows.append(static_row)

                E_list.append(int(group[self.OUTCOME_EVENT_COL].max()))
                T_list.append(float(group[self.OUTCOME_TIME_COL].max()))

        static_df = pd.DataFrame(static_rows).reset_index(drop=True)
        static_df = self._preprocess_static(static_df, fit=fit)

        static_df = static_df.fillna(0)  # ✅ FIX STATIC

        T = np.array(T_list, dtype=float)
        E = np.array(E_list, dtype=float)
        
        # ✅ FIX CRITICO
        T = np.nan_to_num(T, nan=0.0, posinf=0.0, neginf=0.0)
        E = np.nan_to_num(E, nan=0.0)
        
        E = E.astype(int)

        # ✅ DEBUG CHECK (opzionale)
        assert not np.isnan(static_df.values).any(), "NaN nelle static!"
        assert not any(w.isna().any().any() for w in all_windows), "NaN nelle temporal!"

        print(f"Finestre totali: {len(all_windows)}")
        print(f"Pazienti con evento (dec_intra=1): {E.sum()} / {len(E)}")

        loader = TimeSeriesSurvivalDataLoader(
            temporal_data=all_windows,
            observation_times=all_times,
            static_data=static_df,
            T=T,
            E=E,
            static_categorical_columns=self.STATIC_BINARY + self.STATIC_CATEGORICAL,
            temporal_categorical_columns=self.TEMPORAL_BINARY + self.TEMPORAL_CATEGORICAL,
        )

        return loader, patient_ids

In [ ]:
l = RicoveriTimeSeriesDataloader(
      seq_len=5,
      step=5,
      max_windows_per_patient=1,
      balance_deceased=True,   # ← FIX: include anche sopravvissuti
      n_patients=10000,
  )

loader_train, ids = l.load(train_df, fit=True)

In [12]:
from synthcity.utils.serialization import save, load
from synthcity.utils.serialization import save_to_file, load_from_file

synthetic = load_from_file('/kaggle/input/datasets/manuelamarenghi/syntehtic-dataset-dataframe/timegan_10000pz_synthetic.csv')

In [13]:
top_cod = synthetic['seq_id'].unique()[:7500]
synthetic = synthetic[synthetic['seq_id'].isin(top_cod)]

In [14]:
synthetic = synthetic.sort_values(by=['seq_id', 'seq_time_id'])

In [25]:
real = loader_train.dataframe()

In [ ]:
pip install scikit-survival

In [26]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

X_real = real.drop(columns=["seq_out_time_to_event","seq_time_id", "seq_out_event"])
y_real = Surv.from_arrays(real["seq_out_event"], real["seq_out_time_to_event"])

X_syn = synthetic.drop(columns=["seq_out_time_to_event","seq_time_id", "seq_out_event"])
y_syn = Surv.from_arrays(synthetic["seq_out_event"], synthetic["seq_out_time_to_event"])

In [28]:
X_real_train, X_real_test, y_real_train, y_real_test = train_test_split(
    X_real, y_real, test_size=0.3, random_state=42
)

In [22]:
def build_rsf():
    return RandomSurvivalForest(
        n_estimators=100,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=15,
        max_features="sqrt",
        n_jobs=1,
        random_state=42
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sksurv.metrics import concordance_index_censored


# =========================================================
# STYLE (THESIS READY)
# =========================================================

plt.style.use("seaborn-v0_8-whitegrid")

# remove grids globally
plt.rcParams["axes.grid"] = False

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
    "figure.titlesize": 16
})

cmap = cm.get_cmap("viridis", 6)

ratios_syn = [0, 20, 40, 60, 80, 100]
s = 10000


# =========================================================
# STORAGE
# =========================================================

cindexes = []
cindex_stds = []

labels = []


# =========================================================
# HELPERS
# =========================================================

def cv_metrics(X, y, model_fn, n_splits=3):

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    cidx = []

    surv_all = []
    chf_all = []

    global_time_surv = None
    global_time_chf = None

    for tr, te in kf.split(X):

        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]

        # =================================================
        # TRAIN
        # =================================================

        model = model_fn()

        model.fit(X_tr, y_tr)

        # =================================================
        # C-INDEX
        # =================================================

        risk = model.predict(X_te)

        c = concordance_index_censored(
            y_te["event"],
            y_te["time"],
            risk
        )[0]

        cidx.append(c)

        # =================================================
        # SURVIVAL CURVES
        # =================================================

        surv = model.predict_survival_function(X_te[:25])

        if global_time_surv is None:
            global_time_surv = surv[0].x

        surv_interp = []

        for fn in surv:

            interp_vals = np.interp(
                global_time_surv,
                fn.x,
                fn(fn.x)
            )

            surv_interp.append(interp_vals)

        surv_all.append(
            np.mean(surv_interp, axis=0)
        )

        # =================================================
        # CHF CURVES
        # =================================================

        chf = model.predict_cumulative_hazard_function(X_te[:25])

        if global_time_chf is None:
            global_time_chf = chf[0].x

        chf_interp = []

        for fn in chf:

            interp_vals = np.interp(
                global_time_chf,
                fn.x,
                fn(fn.x)
            )

            chf_interp.append(interp_vals)

        chf_all.append(
            np.mean(chf_interp, axis=0)
        )

    return {

        "cindex_mean": np.mean(cidx),

        # SEM instead of STD
        "cindex_std": (
            np.std(cidx) / np.sqrt(len(cidx))
        ),

        "surv_mean": np.mean(surv_all, axis=0),
        "surv_std": np.std(surv_all, axis=0),

        "chf_mean": np.mean(chf_all, axis=0),
        "chf_std": np.std(chf_all, axis=0),

        "t_surv": global_time_surv,
        "t_chf": global_time_chf
    }


# =========================================================
# PERMUTATION IMPORTANCE
# =========================================================

def compute_perm_importance(model, X, y):

    def score(estimator, Xv, yv):

        risk = estimator.predict(Xv)

        return concordance_index_censored(
            yv["event"],
            yv["time"],
            risk
        )[0]

    result = permutation_importance(
        model,
        X,
        y,
        scoring=score,
        n_repeats=3,
        random_state=42,
        n_jobs=-1
    )

    return (
        result.importances_mean,
        result.importances_std
    )


# =========================================================
# MAIN LOOP
# =========================================================

fig_surv, ax_surv = plt.subplots(figsize=(9, 6))
fig_chf, ax_chf = plt.subplots(figsize=(9, 6))

n_total = len(X_real_train)

for i, ratio in enumerate(ratios_syn):

    color = cmap(i)

    r = ratio / 100

    n_syn = int(n_total * r)
    n_real = n_total - n_syn

    # =====================================================
    # REAL
    # =====================================================

    X_real_part = X_real_train.sample(
        n=n_real,
        random_state=42
    )

    y_real_part = y_real_train[
        X_real_train.index.get_indexer(
            X_real_part.index
        )
    ]

    # =====================================================
    # SYNTHETIC
    # =====================================================

    X_syn_part = X_syn.sample(
        n=n_syn,
        random_state=42
    )

    y_syn_part = y_syn[
        X_syn.index.get_indexer(
            X_syn_part.index
        )
    ]

    # =====================================================
    # MIX
    # =====================================================

    X_mix = pd.concat(
        [X_real_part, X_syn_part],
        axis=0
    )

    y_mix = np.concatenate(
        [y_real_part, y_syn_part]
    )

    # =====================================================
    # MODEL FUNCTION
    # =====================================================

    def model_fn():
        return build_rsf()

    # =====================================================
    # CV METRICS
    # =====================================================

    results = cv_metrics(
        X_mix,
        y_mix,
        model_fn
    )

    # =====================================================
    # STORE CINDEX
    # =====================================================

    cindexes.append(
        results["cindex_mean"]
    )

    cindex_stds.append(
        results["cindex_std"]
    )


    label = ""
    if ratio == 95 :
        label = "synthetic"
    elif ratio == 0 :
        label = "real"
    else :
        label = f"{ratio}% synthetic"

    labels.append(label)

    # =====================================================
    # SURVIVAL PLOT
    # =====================================================

    ax_surv.step(
        results["t_surv"],
        results["surv_mean"],
        where="post",
        color=color,
        linewidth=2,
        label=label
    )

    ax_surv.fill_between(
        results["t_surv"],
        results["surv_mean"] - results["surv_std"],
        results["surv_mean"] + results["surv_std"],
        alpha=0.12,
        color=color
    )

    # =====================================================
    # CHF PLOT
    # =====================================================

    ax_chf.step(
        results["t_chf"],
        results["chf_mean"],
        where="post",
        color=color,
        linewidth=2,
        label=label
    )

    ax_chf.fill_between(
        results["t_chf"],
        results["chf_mean"] - results["chf_std"],
        results["chf_mean"] + results["chf_std"],
        alpha=0.12,
        color=color
    )


# =========================================================
# SURVIVAL FIGURE
# =========================================================

ax_surv.set_title(
    "10000p Cross-Validated Survival Curves (RSF)"
)

ax_surv.set_xlabel("Time")

ax_surv.set_ylabel(
    "Survival Probability"
)

ax_surv.spines["top"].set_visible(False)
ax_surv.spines["right"].set_visible(False)

ax_surv.legend(
    title="Synthetic ratio",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
    fancybox=False,
    framealpha=1,
    edgecolor="black"
)

fig_surv.tight_layout()
fig_surv.savefig(
    f"/kaggle/working/surv_mean_rsf_cross_{s}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# CHF FIGURE
# =========================================================

ax_chf.set_title("10000p Cross-Validated Cumulative Hazard (RSF)")
ax_chf.set_xlabel("Time")
ax_chf.set_ylabel("Cumulative Hazard")

ax_chf.spines["top"].set_visible(False)
ax_chf.spines["right"].set_visible(False)

ax_chf.legend(
    title="Synthetic ratio",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
    fancybox=False,
    framealpha=1,
    edgecolor="grey"
)

fig_chf.tight_layout()

# 🔥 FORZA RENDERING
fig_chf.canvas.draw()

fig_chf.savefig(
    f"/kaggle/working/chf_mean_rsf_cross_{s}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# C-INDEX FIGURE
# =========================================================

plt.figure(figsize=(7, 5))

plt.errorbar(
    ratios_syn,
    cindexes,
    yerr=cindex_stds,
    marker="o",
    linewidth=1.8,
    markersize=6,
    capsize=3,
    color="black",
    elinewidth=1
)

plt.xticks(ratios_syn)

plt.xlabel(
    "Synthetic Data Ratio (%)"
)

plt.ylabel("C-index")

plt.title(
    "10000p C-index vs Synthetic Data Ratio (RSF)"
)

plt.ylim(
    min(cindexes) - 0.02,
    max(cindexes) + 0.02
)

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"/kaggle/working/C-index_rsf_cross_{s}.png")

plt.show()


# =========================================================
# FINAL MODEL
# =========================================================

final_model = build_rsf()

final_model.fit(
    X_real_train,
    y_real_train
)

# =========================================================
# PERMUTATION IMPORTANCE
# =========================================================

imp, imp_std = compute_perm_importance(
    final_model,
    X_real_test,
    y_real_test
)

feature_names = np.array(
    X_real_train.columns
)

# =========================================================
# TOP FEATURES
# =========================================================

top_k = 5

idx = np.argsort(imp)[::-1][:top_k]

plt.figure(figsize=(8, 6))

plt.barh(
    feature_names[idx][::-1],
    imp[idx][::-1],
    xerr=imp_std[idx][::-1],
    color="purple"
)

plt.xlabel(
    "Permutation Importance"
)

plt.title(
    "10000p Top 5 Most Important Features (RSF)"
)

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"/kaggle/working/perm_imp_rsf_{s}.png")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index


# =========================================================
# STYLE (THESIS READY)
# =========================================================

plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams["axes.grid"] = False

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
    "figure.titlesize": 16
})

cmap = cm.get_cmap("viridis", 6)

ratios_syn = [0, 20, 40, 60, 80, 100]

s = 10000


# =========================================================
# SCALING
# =========================================================

scaler = StandardScaler()

X_real_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_real_train),
    columns=X_real_train.columns,
    index=X_real_train.index
)

X_real_test_scaled = pd.DataFrame(
    scaler.transform(X_real_test),
    columns=X_real_test.columns,
    index=X_real_test.index
)

X_syn_scaled = pd.DataFrame(
    scaler.transform(X_syn),
    columns=X_syn.columns,
    index=X_syn.index
)


# =========================================================
# STORAGE
# =========================================================

cindexes = []
cindex_stds = []

labels = []


# =========================================================
# HELPERS
# =========================================================

def build_cox_df(X, y):

    df = X.copy()

    df["time"] = y["time"]
    df["event"] = y["event"]

    return df


# =========================================================
# CV METRICS
# =========================================================

def cv_metrics(X, y, n_splits=3):

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    cidx = []

    surv_all = []
    chf_all = []

    global_time_surv = None
    global_time_chf = None

    for tr, te in kf.split(X):

        X_tr, X_te = X.iloc[tr], X.iloc[te]

        y_tr, y_te = y[tr], y[te]

        # =================================================
        # BUILD DATAFRAMES
        # =================================================

        df_train = build_cox_df(X_tr, y_tr)

        df_test = build_cox_df(X_te, y_te)

        # =================================================
        # MODEL
        # =================================================

        model = CoxPHFitter(
            penalizer=0.01
        )

        model.fit(
            df_train,
            duration_col="time",
            event_col="event"
        )

        # =================================================
        # RISK SCORE
        # =================================================

        risk = model.predict_partial_hazard(
            X_te
        ).values.flatten()

        # =================================================
        # C-INDEX
        # =================================================

        c = concordance_index(
            y_te["time"],
            -risk,
            y_te["event"]
        )

        cidx.append(c)

        # =================================================
        # SURVIVAL CURVES
        # =================================================

        surv = model.predict_survival_function(
            X_te[:25]
        )

        surv_mean = surv.mean(axis=1)

        if global_time_surv is None:

            global_time_surv = surv.index.values

        surv_interp = np.interp(
            global_time_surv,
            surv.index.values,
            surv_mean.values
        )

        surv_all.append(surv_interp)

        # =================================================
        # CHF CURVES
        # =================================================

        chf = model.predict_cumulative_hazard(
            X_te[:25]
        )

        chf_mean = chf.mean(axis=1)

        if global_time_chf is None:

            global_time_chf = chf.index.values

        chf_interp = np.interp(
            global_time_chf,
            chf.index.values,
            chf_mean.values
        )

        chf_all.append(chf_interp)

    return {

        "cindex_mean": np.mean(cidx),

        "cindex_std": (
            np.std(cidx) / np.sqrt(len(cidx))
        ),

        "surv_mean": np.mean(surv_all, axis=0),
        "surv_std": np.std(surv_all, axis=0),

        "chf_mean": np.mean(chf_all, axis=0),
        "chf_std": np.std(chf_all, axis=0),

        "t_surv": global_time_surv,
        "t_chf": global_time_chf
    }


# =========================================================
# PERMUTATION IMPORTANCE
# =========================================================

def compute_perm_importance(model, X, y):

    baseline_risk = model.predict_partial_hazard(
        X
    ).values.flatten()

    baseline_score = concordance_index(
        y["time"],
        -baseline_risk,
        y["event"]
    )

    importances = []
    importances_std = []

    rng = np.random.default_rng(42)

    for col in X.columns:

        scores = []

        for _ in range(3):

            X_perm = X.copy()

            X_perm[col] = rng.permutation(
                X_perm[col].values
            )

            risk = model.predict_partial_hazard(
                X_perm
            ).values.flatten()

            score = concordance_index(
                y["time"],
                -risk,
                y["event"]
            )

            scores.append(
                baseline_score - score
            )

        importances.append(
            np.mean(scores)
        )

        importances_std.append(
            np.std(scores)
        )

    return (
        np.array(importances),
        np.array(importances_std)
    )


# =========================================================
# MAIN LOOP
# =========================================================

fig_surv, ax_surv = plt.subplots(figsize=(9, 6))
fig_chf, ax_chf = plt.subplots(figsize=(9, 6))

n_total = len(X_real_train_scaled)

for i, ratio in enumerate(ratios_syn):

    color = cmap(i)

    r = ratio / 100

    n_syn = int(n_total * r)
    n_real = n_total - n_syn

    # =====================================================
    # REAL
    # =====================================================

    X_real_part = X_real_train_scaled.sample(
        n=n_real,
        random_state=42
    )

    y_real_part = y_real_train[
        X_real_train_scaled.index.get_indexer(
            X_real_part.index
        )
    ]

    # =====================================================
    # SYNTHETIC
    # =====================================================

    X_syn_part = X_syn_scaled.sample(
        n=n_syn,
        random_state=42
    )

    y_syn_part = y_syn[
        X_syn_scaled.index.get_indexer(
            X_syn_part.index
        )
    ]

    # =====================================================
    # MIX
    # =====================================================

    X_mix = pd.concat(
        [X_real_part, X_syn_part],
        axis=0
    )

    y_mix = np.concatenate(
        [y_real_part, y_syn_part]
    )

    # =====================================================
    # CV METRICS
    # =====================================================

    results = cv_metrics(
        X_mix,
        y_mix
    )

    # =====================================================
    # STORE
    # =====================================================

    cindexes.append(
        results["cindex_mean"]
    )

    cindex_stds.append(
        results["cindex_std"]
    )

    label = ""
    if ratio == 0:
        label = "real"
    elif ratio == 95:
        label = "synthetic"
    else:
        label = f"{ratio}% synthetic"

    labels.append(label)

    # =====================================================
    # SURVIVAL PLOT
    # =====================================================

    ax_surv.step(
        results["t_surv"],
        results["surv_mean"],
        where="post",
        color=color,
        linewidth=2,
        label=label
    )

    ax_surv.fill_between(
        results["t_surv"],
        results["surv_mean"] - results["surv_std"],
        results["surv_mean"] + results["surv_std"],
        alpha=0.12,
        color=color
    )

    # =====================================================
    # CHF PLOT
    # =====================================================

    ax_chf.step(
        results["t_chf"],
        results["chf_mean"],
        where="post",
        color=color,
        linewidth=2,
        label=label
    )

    ax_chf.fill_between(
        results["t_chf"],
        results["chf_mean"] - results["chf_std"],
        results["chf_mean"] + results["chf_std"],
        alpha=0.12,
        color=color
    )


# =========================================================
# SURVIVAL FIGURE
# =========================================================

ax_surv.set_title(
    "10000p Cross-Validated Survival Curves (CoxPH)"
)

ax_surv.set_xlabel("Time")

ax_surv.set_ylabel(
    "Survival Probability"
)

ax_surv.spines["top"].set_visible(False)
ax_surv.spines["right"].set_visible(False)

ax_surv.legend(
    title="Synthetic ratio",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
    fancybox=False,
    framealpha=1,
    edgecolor="black"
)

fig_surv.tight_layout()

fig_surv.savefig(
    f"/kaggle/working/surv_mean_coxph_{s}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# CHF FIGURE
# =========================================================

ax_chf.set_title(
    "10000p Cross-Validated Cumulative Hazard (CoxPH)"
)

ax_chf.set_xlabel("Time")

ax_chf.set_ylabel(
    "Cumulative Hazard"
)

ax_chf.spines["top"].set_visible(False)
ax_chf.spines["right"].set_visible(False)

ax_chf.legend(
    title="Synthetic ratio",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
    fancybox=False,
    framealpha=1,
    edgecolor="black"
)

fig_chf.tight_layout()

fig_chf.savefig(
    f"/kaggle/working/chf_mean_coxph_{s}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# C-INDEX FIGURE
# =========================================================

plt.figure(figsize=(7, 5))

plt.errorbar(
    ratios_syn,
    cindexes,
    yerr=cindex_stds,
    marker="o",
    linewidth=1.8,
    markersize=6,
    capsize=3,
    color="black",
    elinewidth=1
)

plt.xticks(ratios_syn)

plt.xlabel(
    "Synthetic Data Ratio (%)"
)

plt.ylabel("C-index")

plt.title(
    "10000p C-index vs Synthetic Data Ratio (CoxPH)"
)

plt.ylim(
    min(cindexes) - 0.02,
    max(cindexes) + 0.02
)

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    f"/kaggle/working/C-index_coxph_{s}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# FINAL MODEL
# =========================================================

final_model = CoxPHFitter(
    penalizer=0.01
)

df_final = build_cox_df(
    X_real_train_scaled,
    y_real_train
)

final_model.fit(
    df_final,
    duration_col="time",
    event_col="event"
)


# =========================================================
# PERMUTATION IMPORTANCE
# =========================================================

imp, imp_std = compute_perm_importance(
    final_model,
    X_real_test_scaled,
    y_real_test
)

feature_names = np.array(
    X_real_train_scaled.columns
)

top_k = 5

idx = np.argsort(imp)[::-1][:top_k]

plt.figure(figsize=(8, 6))

plt.barh(
    feature_names[idx][::-1],
    imp[idx][::-1],
    xerr=imp_std[idx][::-1],
    color="purple"
)

plt.xlabel(
    "Permutation Importance"
)

plt.title(
    "10000p Top 5 Most Important Features (CoxPH)"
)

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    f"/kaggle/working/perm_imp_coxph_{s}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()